In [1]:
import os
import pandas as pd

# Check files
print("Files in directory:", os.listdir())

# Rename if needed
if "orders (9).csv" in os.listdir():
    os.rename("orders (9).csv", "orders.csv")

if "churn_labels (9).csv" in os.listdir():
    os.rename("churn_labels (9).csv", "churn_labels.csv")

# Load data
orders = pd.read_csv("orders.csv")
churn = pd.read_csv("churn_labels.csv")

print("✅ Files loaded successfully")

# -----------------------------
# DATE FORMATTING
# -----------------------------
orders['order_date'] = pd.to_datetime(orders['order_date'])
churn['snapshot_date'] = pd.to_datetime(churn['snapshot_date'])

# -----------------------------
# FILTER DATA (NO LEAKAGE)
# -----------------------------
orders = orders.merge(churn[['customer_id','snapshot_date']], on='customer_id', how='left')
orders = orders[orders['order_date'] <= orders['snapshot_date']]

# -----------------------------
# RFM FEATURE CREATION
# -----------------------------
rfm = orders.groupby('customer_id').agg({
    'order_id': 'count',
    'gross_amount': 'sum',
    'order_date': 'max'
}).reset_index()

rfm.columns = ['customer_id', 'frequency', 'monetary', 'last_order_date']

# Merge snapshot date
rfm = rfm.merge(churn[['customer_id','snapshot_date']], on='customer_id', how='left')

# Recency calculation
rfm['recency'] = (rfm['snapshot_date'] - rfm['last_order_date']).dt.days

# -----------------------------
# HANDLE MISSING VALUES
# -----------------------------
rfm.fillna(0, inplace=True)

# -----------------------------
# FEATURE SCALING
# -----------------------------
X = rfm[['recency','frequency','monetary']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# KMEANS CLUSTERING
# -----------------------------
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['cluster'] = kmeans.fit_predict(X_scaled)

# -----------------------------
# SEGMENT ASSIGNMENT LOGIC
# -----------------------------
# Sort clusters based on business meaning
cluster_summary = rfm.groupby('cluster').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).reset_index()

# Sort by best customers (low recency, high frequency, high monetary)
cluster_summary = cluster_summary.sort_values(
    by=['recency','frequency','monetary'],
    ascending=[True, False, False]
).reset_index(drop=True)

# Assign segment names dynamically
segment_labels = ['High Value', 'Regular', 'At Risk', 'Lost']

cluster_summary['segment_name'] = segment_labels

# Merge back segment names
rfm = rfm.merge(cluster_summary[['cluster','segment_name']], on='cluster', how='left')

# -----------------------------
# FINAL OUTPUT (segments.csv)
# -----------------------------
final_segments = rfm[[
    'customer_id',
    'recency',
    'frequency',
    'monetary',
    'segment_name'
]]

# Save CSV
final_segments.to_csv('segments.csv', index=False)

# Preview
print("✅ segments.csv generated successfully")
print(final_segments.head())

In file included from <<< inputs >>>:1:
input_line_2:1:10: error: expected ';' after module name
    1 | import os
      |          ^
      |          ;
input_line_2:1:8: fatal error: module 'os' not found
    1 | import os
      | ~~~~~~~^~
Failed to parse via ::process:Parsing failed.


Error: : Compilation error! In file included from <<< inputs >>>:1:
input_line_2:1:10: error: expected ';' after module name
    1 | import os
      |          ^
      |          ;
input_line_2:1:8: fatal error: module 'os' not found
    1 | import os
      | ~~~~~~~^~
Failed to parse via ::process:Parsing failed.
